In [1]:
import numpy as np
import re
import pandas as pd
import seaborn as sns

In [2]:
df = pd.read_csv('cars.csv')

In [3]:
df.dtypes

manufacturer            object
model                   object
year                     int64
mileage                float64
engine                  object
transmission            object
drivetrain              object
fuel_type               object
mpg                     object
exterior_color          object
interior_color          object
accidents_or_damage    float64
one_owner              float64
personal_use_only      float64
seller_name             object
seller_rating          float64
driver_rating          float64
driver_reviews_num     float64
price_drop             float64
price                  float64
dtype: object

In [4]:
df.dtypes.to_dict()

{'manufacturer': dtype('O'),
 'model': dtype('O'),
 'year': dtype('int64'),
 'mileage': dtype('float64'),
 'engine': dtype('O'),
 'transmission': dtype('O'),
 'drivetrain': dtype('O'),
 'fuel_type': dtype('O'),
 'mpg': dtype('O'),
 'exterior_color': dtype('O'),
 'interior_color': dtype('O'),
 'accidents_or_damage': dtype('float64'),
 'one_owner': dtype('float64'),
 'personal_use_only': dtype('float64'),
 'seller_name': dtype('O'),
 'seller_rating': dtype('float64'),
 'driver_rating': dtype('float64'),
 'driver_reviews_num': dtype('float64'),
 'price_drop': dtype('float64'),
 'price': dtype('float64')}

# VALORES ERRÓNEOS

In [5]:
pd.isnull(df).sum()

manufacturer                0
model                       0
year                        0
mileage                   506
engine                  15050
transmission             9904
drivetrain              21562
fuel_type               22927
mpg                    142071
exterior_color           8859
interior_color          56975
accidents_or_damage     24212
one_owner               31483
personal_use_only       24852
seller_name              8593
seller_rating          213973
driver_rating           31632
driver_reviews_num          0
price_drop             351979
price                       0
dtype: int64

In [6]:
pd.isnull(df[[x for x in df.columns[pd.isnull(df).any()].tolist()]]).sum()

mileage                   506
engine                  15050
transmission             9904
drivetrain              21562
fuel_type               22927
mpg                    142071
exterior_color           8859
interior_color          56975
accidents_or_damage     24212
one_owner               31483
personal_use_only       24852
seller_name              8593
seller_rating          213973
driver_rating           31632
price_drop             351979
dtype: int64

In [7]:
df[df.isna().any(axis=1)][[x for x in df.columns[pd.isnull(df).any()]]]

,mileage,engine,transmission,drivetrain,fuel_type,mpg,exterior_color,interior_color,accidents_or_damage,one_owner,personal_use_only,seller_name,seller_rating,driver_rating,price_drop
0,92945.0,"1.5L I-4 i-VTEC variable valve control, engine...",Automatic,Front-wheel Drive,Gasoline,39-38,Black,Parchment,0.0,0.0,0.0,Iconic Coach,NaN,4.4,300.0
1,47645.0,1.5L I4 8V MPFI SOHC Hybrid,Automatic CVT,Front-wheel Drive,Hybrid,39-38,Gray,Ebony,1.0,1.0,1.0,Kars Today,NaN,4.4,NaN
3,117598.0,1.5L I4 8V MPFI SOHC Hybrid,Automatic CVT,Front-wheel Drive,Hybrid,39-38,Polished Metal Metallic,NaN,0.0,1.0,1.0,Apple Tree Acura,NaN,4.4,675.0
4,114865.0,1.5L I4 8V MPFI SOHC Hybrid,Automatic CVT,Front-wheel Drive,Hybrid,39-38,NaN,Ebony,1.0,0.0,1.0,Herb Connolly Chevrolet,3.7,4.4,300.0
6,57212.0,1.5L I4 8V MPFI SOHC Hybrid,Automatic CVT,Front-wheel Drive,Hybrid,39-38,Silver,Ebony,0.0,1.0,1.0,Ohio Car Mart,NaN,4.4,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
762082,6302.0,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,All-wheel Drive,Gasoline,25-33,Birch Light Metallic,Charcoal,0.0,1.0,0.0,Scott Volvo Cars Allentown,NaN,4.2,NaN
762083,13972.0,2.0L I4 16V GDI DOHC,8-Speed Automatic,All-wheel Drive,Gasoline,21-32,White,NaN,0.0,1.0,0.0,Simmons-Rockwell Hyundai,NaN,4.9,NaN
762087,72900.0,250.0HP 2.5L 5 Cylinder Engine Gasoline Fuel,A/T,Front-wheel Drive,Gasoline,NaN,Red,Beige,NaN,NaN,NaN,NaN,NaN,4.5,NaN
762088,92000.0,2.5L I5 20V MPFI DOHC Turbo,6-Speed Automatic,Front-wheel Drive,Gasoline,21-30,Ice White,Soft Beige,0.0,0.0,1.0,Dapper Car Sales,NaN,4.8,300.0


# Mileage
#### Hemos imputado los valores faltantes de mileage usando una estrategia escalonada, rellenando primero por manufacturer + model + year y después por medianas de fabricante y globales. Así hemos obtenido un kilometraje coherente con la edad y el tipo de coche, evitando perder datos valiosos.

In [8]:
df['mileage'] = df.groupby(['manufacturer','model','year'])['mileage'].transform(
    lambda x: x.fillna(x.median())
)

#Estamos rellenando los valores nulos con la mediana teniendo en cuenta las columnas: 'manufacturer','model','year'.

In [9]:
df['mileage'] = df['mileage'].fillna(df['mileage'].median())
#Tras rellenar los ausentes de 'mileage' con la mediana de 'manufacturer','model','year', nos siguen quedando 20 valores nulos. A estos últimos le hemos rellenado los valores teniendo en cuenta la mediana de 'mileage'.

In [10]:
df['mileage'].describe()

count    7.620910e+05
mean     5.577397e+04
std      4.356073e+04
min      0.000000e+00
25%      2.327400e+04
50%      4.559000e+04
75%      7.836300e+04
max      1.119067e+06
Name: mileage, dtype: float64

In [11]:
outliers = df[df['mileage'] > 400000]
outliers[['year','manufacturer','model','mileage']] \
    .sort_values(by='mileage', ascending=False).head(20)

,year,manufacturer,model,mileage
756847,2010,Volvo,XC70 3.2L,1119067.0
494409,1967,Lincoln,Continental,999999.0
76422,1998,Buick,Century Custom,999999.0
511961,2011,Mercedes-Benz,GLK-Class GLK 350,999999.0
617503,1959,Porsche,356 A,974302.0
319941,2013,Honda,Civic LX,938032.0
522994,2021,Mercedes-Benz,E-Class E 350 4MATIC,915383.0
136180,2013,Chevrolet,Silverado 3500 LTZ,777698.0
669643,2020,Toyota,Camry LE,769938.0
229883,2014,Ford,F-350 Lariat Super Duty,763474.0


In [12]:
#Tras revisar los outliers de mileage, hay que saber interpretarlos. Vemos algunos registros que no son nada fiables. Por ejemplo, un Toyota Camry LE del 2020 con 769.938 km.
#También vemos un Porsche 359 A de 1959 con un 1.000.0000 de km aprox. Hemos decidido eliminar los registros cuyo 'year' sea superior al 2017 y tengan más de 400.000 km. 
#Es muy complicado hacerle tanto km en tan pocos años, ni siquiera un taxista o un camionero llega a esos números.
#También vamos a eliminar los coches cuyo 'year' sea inferior a 1990 y tengan más de 400.000 km ya que los coches de los 60-70-80 no tenían motores capaces de ser tan duraderos.
#En la década de los 90, se comienzan a fabricar coches con motores potentes, de los cuales algunos han llegado posteriormente al millón de km con un mantenimiento adecuado.

to_delete_1 = df[(df['mileage'] > 400000) & (df['year'] > 2015)]
to_delete_2 = df[(df['mileage'] > 400000) & (df['year'] < 1990)]

to_delete = pd.concat([to_delete_1, to_delete_2])

df = df.drop(to_delete.index)

# Engine
#### Hemos limpiado la columna engine dividiéndola en tres variables numéricas y categóricas: engine_liters, engine_cylinders y engine_type. Después hemos imputado los valores faltantes de litros y cilindros usando medianas por grupos coherentes (manufacturer + model + year + cylinders), quedando listas para ML

In [13]:
#Pasamos todos los valores de 'engine' a minúsculas.
df['engine_clean'] = df['engine'].str.lower()

In [14]:
#Extraemos la cilindrada (litros)
def extract_liters(x):
    if pd.isna(x):
        return None
    match = re.search(r'(\d\.\d|\d)\s*l', x)
    if match:
        return float(match.group(1))
    return None

df['engine_liters'] = df['engine_clean'].apply(extract_liters)

In [15]:
#Extraemos el número de cilindridos
def extract_cylinders(x):
    if pd.isna(x):
        return None
    match = re.search(r'(v|i|l)(\d)', x)
    if match:
        return int(match.group(2))
    return None

df['engine_cylinders'] = df['engine_clean'].apply(extract_cylinders)

In [16]:
#Extraemos el tipo de motor
def extract_engine_type(x):
    if pd.isna(x):
        return "unknown"
    if "turbo" in x or "t" in x:
        return "turbo"
    if "super" in x:
        return "supercharged"
    if "electric" in x:
        return "electric"
    if "hybrid" in x:
        return "hybrid"
    return "natural"

df['engine_type'] = df['engine_clean'].apply(extract_engine_type)

In [17]:
#Si engine es electric: 
#litros=0; 
#cilindros=0
#Si creamos estas 3 columnas para diferenciar realmente entre la cilindrada, el número de cilindros y el tipo de motor podemos eliminar posteriormente 'engine_clean':
df.drop(columns=['engine_clean'], inplace=True)

In [18]:
df['engine_liters'] = df.groupby(['manufacturer','model'])['engine_liters'] \
                         .transform(lambda x: x.fillna(x.median()))
#A los missing values de la nueva columna 'engine_liters', le hemos imputado la mediana teniendo en cuenta las columnas 'manufacturer' y 'model'. 

In [19]:
df['engine_liters'] = df.groupby(['manufacturer'])['engine_liters'] \
                         .transform(lambda x: x.fillna(x.median()))
#Como nos siguen quedan valores nulos después de la imputación anterior, le hemos imputado de nuevo la mediana pero ahora teniendo solo en cuenta 'manufacturer'.

In [20]:
df['engine_cylinders'] = df.groupby(['manufacturer','model'])['engine_cylinders'] \
                            .transform(lambda x: x.fillna(x.median()))

#A los missing values de la nueva columna 'engine_cylinders', le hemos imputado la mediana teniendo en cuenta las columnas 'manufacturer' y 'model'. 

In [21]:
df['engine_cylinders'] = df.groupby(['manufacturer'])['engine_cylinders'] \
                            .transform(lambda x: x.fillna(x.median()))
#Como nos siguen quedan valores nulos después de la imputación anterior, le hemos imputado de nuevo la mediana pero ahora teniendo solo en cuenta 'manufacturer'.

In [22]:
df.drop(columns=['engine'], inplace=True)

# transmission
#### Hemos clasificado y unificado todos los valores de transmission en automatic, manual y unknown, detectando variantes como A/T, CVT, Dual-Clutch, 8-speed, etc., y hemos eliminado los registros clasificados como “other” tras limpiar la columna correctamente.

In [23]:
df['transmission_clean'] = df['transmission'].astype(str).str.lower()

def simplify_transmission(x):
    if pd.isna(x) or x.strip() == "":
        return "unknown"
    
    # AUTOMÁTICAS
    if any(word in x for word in [
        "automatic",          
        "auto ", " auto",     
        "a/t", "at",
        "cvt", 
        "continuously variable", 
        "variable",
        "dct", 
        "dual", 
        "tiptronic", 
        "dsg",
        "shift",
        "8-speed", "8 speed", "8spd",
        "7-speed", "7 speed", "7spd",
        "9-speed", "9 speed", "9spd",
        "10-speed", "10 speed", "10spd"
    ]):
        return "automatic"
    
    # MANUALES
    if any(word in x for word in [
        "manual",
        "m/t", "mt",
        "man ",
        "5-speed", "5 speed",
        "6-speed", "6 speed",
        "5spd", "6spd"
    ]):
        return "manual"
    
    return "other"

df['transmission_simple'] = df['transmission_clean'].apply(simplify_transmission)
df = df.drop(columns=['transmission_clean'])


In [24]:
df = df[df['transmission_simple'] != 'other']

In [25]:
df.drop(columns=['transmission'], inplace=True)

# drivetrain
#### Hemos normalizado todos los valores de drivetrain en las 4 categorías principales (Front-wheel, Rear-wheel, All-wheel, Four-wheel Drive), y eliminado los registros “unknown” y “other” para quedarnos solo con configuraciones de tracción válidas.

In [26]:
df['drivetrain'].value_counts()

drivetrain
Front-wheel Drive                                              239322
All-wheel Drive                                                229454
Four-wheel Drive                                               156058
Rear-wheel Drive                                                96427
FWD                                                              6405
AWD                                                              3579
4WD                                                              1908
RWD                                                              1736
Unknown                                                            91
Front-Wheel Drive                                                  82
All-Wheel Drive                                                    68
Front-Wheel Drive with Limited-Slip Differential                   44
Four-Wheel Drive with Locking and Limited-Slip Differential        42
All-Wheel Drive with Locking and Limited-Slip Differential         33
Four Whee

In [27]:
df['drivetrain_clean'] = df['drivetrain'].astype(str).str.lower()

def simplify_drivetrain(x):
    if pd.isna(x) or x.strip() == "" or x == "nan":
        return "unknown" 

    # FRONT-WHEEL DRIVE (FWD)
    if any(word in x for word in [
        "front-wheel", "front wheel", "fwd"
    ]):
        return "Front-wheel Drive"
    
    # REAR-WHEEL DRIVE (RWD)
    if any(word in x for word in [
        "rear-wheel", "rear wheel", "rwd"
    ]):
        return "Rear-wheel Drive"
    
    # ALL-WHEEL DRIVE (AWD)
    if any(word in x for word in [
        "all-wheel", "all wheel", "awd"
    ]):
        return "All-wheel Drive"
    
    # FOUR-WHEEL DRIVE (4WD)
    if any(word in x for word in [
        "four-wheel", "four wheel",
        "4wd", "4x4", "4 x 4"
    ]):
        return "Four-wheel Drive"
    
    return "other"

df['drivetrain_simple'] = df['drivetrain_clean'].apply(simplify_drivetrain)
df = df.drop(columns=['drivetrain_clean'])

In [28]:
df = df[df['drivetrain_simple'] != 'unknown']
df = df[df['drivetrain_simple'] != 'other']

In [29]:
df['drivetrain_simple'].value_counts()

drivetrain_simple
Front-wheel Drive    245870
All-wheel Drive      233164
Four-wheel Drive     158062
Rear-wheel Drive      98203
Name: count, dtype: int64

In [30]:
df.drop(columns=['drivetrain'], inplace=True)

# fuel_type
#### Con fuel_type hemos estandarizado todos los valores en 4 categorías limpias (Gasoline, Diesel, Hybrid, Electric),y¡ y hemos eliminado los registros “unknown” y “other” para quedarnos solo con combustibles válidos y consistentes para el modelo.

In [31]:
df['fuel_clean'] = df['fuel_type'].astype(str).str.lower()

def simplify_fuel(x):
    if pd.isna(x) or x.strip() == "" or x == "nan":
        return "unknown"
    
    # GASOLINE
    if any(word in x for word in [
        "gas", "gasoline", 
        "unleaded", "regular", "petrol"
    ]):
        return "Gasoline"
    
    # DIESEL
    if any(word in x for word in [
        "diesel", "tdi", "dci", "crdi"
    ]):
        return "Diesel"
    
    # HYBRID
    if any(word in x for word in [
        "hybrid", "phev", "plug-in"
    ]):
        return "Hybrid"
    
    # ELECTRIC
    if any(word in x for word in [
        "electric", "ev", "battery"
    ]):
        return "Electric"
    
    return "other"

df['fuel_simple'] = df['fuel_clean'].apply(simplify_fuel)
df = df.drop(columns=['fuel_clean'])

In [32]:
df['fuel_simple'].value_counts()

fuel_simple
Gasoline    635130
Hybrid       28720
Diesel       27720
other        19063
Electric     15974
unknown       8692
Name: count, dtype: int64

In [33]:
df = df[(df['fuel_simple'] != 'unknown') &
        (df['fuel_simple'] != 'other')]

# mpg
#### Hemos convertido los rangos de mpg en dos columnas numéricas (mpg_min y mpg_max), corrigiendo incluso los rangos invertidos.
#### Después hemos imputado sus valores faltantes usando medianas por grupos coherentes (manufacturer + model + engine_cylinders)

In [34]:
def parse_ordered_mpg(x):
    if pd.isna(x):
        return (np.nan, np.nan)
    
    x = str(x).strip().replace("–", "-")
    
    # Si no hay rango (solo un valor)
    if "-" not in x:
        try:
            v = float(x)
            return (v, v)
        except:
            return (np.nan, np.nan)
    
    # Extraer números del rango
    nums = re.findall(r'\d+', x)
    
    if len(nums) == 2:
        a, b = float(nums[0]), float(nums[1])
        return (min(a, b), max(a, b))   # ← ORDENAMOS AQUÍ
    
    return (np.nan, np.nan)

df[['mpg_min', 'mpg_max']] = df['mpg'].apply(lambda x: pd.Series(parse_ordered_mpg(x)))

In [35]:
df[['mpg', 'mpg_min', 'mpg_max']].head(20)

,mpg,mpg_min,mpg_max
0,39-38,38.0,39.0
1,39-38,38.0,39.0
2,39-38,38.0,39.0
3,39-38,38.0,39.0
4,39-38,38.0,39.0
5,39-38,38.0,39.0
6,39-38,38.0,39.0
7,NaN,NaN,NaN
8,39-38,38.0,39.0
10,21-22,21.0,22.0


In [36]:
for col in ['mpg_min', 'mpg_max']:
    df[col] = df.groupby(
        ['manufacturer','model','engine_cylinders']
    )[col].transform(lambda x: x.fillna(x.median()))

#A los missing values de las nuevas columnas 'mpg_min y mpg_max', le hemos imputado la mediana teniendo en cuenta las columnas 'manufacturer', 'model' y 'engine_cylinders'.

In [37]:
for col in ['mpg_min', 'mpg_max']:
    df[col] = df.groupby('engine_cylinders')[col] \
                .transform(lambda x: x.fillna(x.median()))

# Como nos siguen quedan valores nulos después de la imputación anterior, le hemos imputado de nuevo la mediana pero ahora teniendo solo en cuenta 'engine_cylinders'

In [38]:
df['mpg_min'] = df['mpg_min'].fillna(df['mpg_min'].median())
df['mpg_max'] = df['mpg_max'].fillna(df['mpg_max'].median())

#Tras rellenar los ausentes de 'mpg_min' y 'mpg_max' con la mediana de manufacturer','model','engine_cylinders', nos siguen quedando valores nulos, pero muy pocos. A estos últimos le hemos rellenado los valores teniendo en cuenta la mediana en general.

In [39]:
df = df.drop(columns=['mpg'])

### Columnas de colores

In [40]:
df = df.drop(columns=['exterior_color', 'interior_color'])
# hemos decidido eliminar las columnas de los colores ya que no tienen un aporte significativo a la hora de predcir el precio, el cual es nuestro objetivo.

## Para los valores nulos de las columnas: 
### 'accidents_or_damage, one_owner y personal_use_only', al ser columnas binarias, hemos decidido imputar los valores nulos siguiendo la distribución ya existente de la variable. Calculando las distribuciones e imputando después

#### accidents_or_damage

In [41]:
dist = df['accidents_or_damage'].value_counts(normalize=True)
dist

accidents_or_damage
0.0    0.769639
1.0    0.230361
Name: proportion, dtype: float64

In [42]:
df['accidents_or_damage'] = df['accidents_or_damage'].fillna(
    np.random.choice(dist.index, p=dist.values)
)

# one_owner

In [43]:
dist1 = df['one_owner'].value_counts(normalize=True)
dist1

one_owner
1.0    0.562455
0.0    0.437545
Name: proportion, dtype: float64

In [44]:
df['one_owner'] = df['one_owner'].fillna(
    np.random.choice(dist.index, p=dist.values)
)

# personal_use_only

In [45]:
dist2 = df['personal_use_only'].value_counts(normalize=True)
dist2

personal_use_only
1.0    0.662176
0.0    0.337824
Name: proportion, dtype: float64

In [46]:
df['personal_use_only'] = df['personal_use_only'].fillna(
    np.random.choice(dist.index, p=dist.values)
)